In [2]:
!pip install tensorflow

   ---------------------------------------- 0.0/350.9 MB ? eta -:--:--
    --------------------------------------- 6.3/350.9 MB 42.8 MB/s eta 0:00:09
    --------------------------------------- 6.3/350.9 MB 42.8 MB/s eta 0:00:09
    --------------------------------------- 7.1/350.9 MB 13.2 MB/s eta 0:00:27
   -- ------------------------------------- 19.1/350.9 MB 23.7 MB/s eta 0:00:15
   --- ------------------------------------ 34.3/350.9 MB 34.1 MB/s eta 0:00:10
   ----- ---------------------------------- 50.1/350.9 MB 40.9 MB/s eta 0:00:08
   ------ --------------------------------- 59.0/350.9 MB 40.9 MB/s eta 0:00:08
   -------- ------------------------------- 71.0/350.9 MB 42.7 MB/s eta 0:00:07
   -------- ------------------------------- 75.8/350.9 MB 40.3 MB/s eta 0:00:07
   --------- ------------------------------ 83.6/350.9 MB 40.1 MB/s eta 0:00:07
   ---------- ----------------------------- 91.8/350.9 MB 39.8 MB/s eta 0:00:07
   ----------- ---------------------------- 99.6/350

### Imports

In [3]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
from sklearn.ensemble import RandomForestClassifier

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, BatchNormalization
from tensorflow.keras.callbacks import EarlyStopping


c:\Users\Olive\AppData\Local\Programs\Python\Python312\Lib\site-packages\requests\__init__.py:113: RequestsDependencyWarning: urllib3 (2.2.1) or chardet (7.3.0)/charset_normalizer (3.3.2) doesn't match a supported version!
  warnings.warn(


In [4]:
url = "https://archive.ics.uci.edu/ml/machine-learning-databases/adult/adult.data"

cols = [
    "age","workclass","fnlwgt","education","education-num",
    "marital-status","occupation","relationship","race",
    "sex","capital-gain","capital-loss","hours-per-week",
    "native-country","income"
]

df = pd.read_csv(url, names=cols)


### Cleaning

In [5]:
df = df.apply(lambda x: x.str.strip() if x.dtype == "object" else x)

In [6]:
# remove missing '?'
df = df.replace('?', np.nan)
df = df.dropna()

In [7]:
df["income"] = df["income"].map({"<=50K": 0, ">50K": 1})

### Feature Engineering

In [8]:
df["hours_per_age"] = df["hours-per-week"] / df["age"]

In [9]:
df = pd.get_dummies(df, drop_first=True)

X = df.drop("income", axis=1)
y = df["income"]

In [10]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

Scale

In [11]:
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

### Model 1 - Deep Learning

In [12]:
model = Sequential()

model.add(Dense(128, activation='relu', input_dim=X_train.shape[1]))
model.add(BatchNormalization())
model.add(Dropout(0.4))

c:\Users\Olive\AppData\Local\Programs\Python\Python312\Lib\site-packages\keras\src\layers\core\dense.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [13]:
model.add(Dense(64, activation='relu'))
model.add(Dropout(0.3))

In [14]:
model.add(Dense(32, activation='relu'))
model.add(Dense(1, activation='sigmoid'))

In [15]:
model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

In [16]:
# Early stopping (diferencial)
early_stop = EarlyStopping(
    monitor='val_loss',
    patience=5,
    restore_best_weights=True
)

In [17]:
history = model.fit(
    X_train, y_train,
    validation_split=0.2,
    epochs=50,
    batch_size=32,
    callbacks=[early_stop],
    verbose=1
)

Epoch 1/50
604/604 ━━━━━━━━━━━━━━━━━━━━ 8s 5ms/step - accuracy: 0.8081 - loss: 0.4092 - val_accuracy: 0.8375 - val_loss: 0.3494
Epoch 2/50
604/604 ━━━━━━━━━━━━━━━━━━━━ 3s 5ms/step - accuracy: 0.8276 - loss: 0.3634 - val_accuracy: 0.8380 - val_loss: 0.3471
Epoch 3/50
604/604 ━━━━━━━━━━━━━━━━━━━━ 3s 5ms/step - accuracy: 0.8324 - loss: 0.3536 - val_accuracy: 0.8438 - val_loss: 0.3393
Epoch 4/50
604/604 ━━━━━━━━━━━━━━━━━━━━ 3s 5ms/step - accuracy: 0.8407 - loss: 0.3444 - val_accuracy: 0.8450 - val_loss: 0.3374
Epoch 5/50
604/604 ━━━━━━━━━━━━━━━━━━━━ 3s 5ms/step - accuracy: 0.8413 - loss: 0.3386 - val_accuracy: 0.8465 - val_loss: 0.3346
Epoch 6/50
604/604 ━━━━━━━━━━━━━━━━━━━━ 3s 4ms/step - accuracy: 0.8452 - loss: 0.3333 - val_accuracy: 0.8494 - val_loss: 0.3341
Epoch 7/50
604/604 ━━━━━━━━━━━━━━━━━━━━ 3s 5ms/step - accuracy: 0.8434 - loss: 0.3308 - val_accuracy: 0.8465 - val_loss: 0.3309
Epoch 8/50
604/604 ━━━━━━━━━━━━━━━━━━━━ 3s 5ms/step - accuracy: 0.8440 - loss: 0.3280 - val_accuracy: 0.

### Evaluation NN

In [18]:
y_pred_nn = (model.predict(X_test) > 0.5).astype(int)

acc_nn = accuracy_score(y_test, y_pred_nn)

print("\n=== Neural Network ===")
print(f"Acurácia: {acc_nn:.4f}")
print(classification_report(y_test, y_pred_nn))

189/189 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step

=== Neural Network ===
Acurácia: 0.8548
              precision    recall  f1-score   support

           0       0.89      0.92      0.90      4503
           1       0.74      0.66      0.70      1530

    accuracy                           0.85      6033
   macro avg       0.81      0.79      0.80      6033
weighted avg       0.85      0.85      0.85      6033

